In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
import gsw
import matplotlib.pyplot as plt
from scipy.stats import linregress


# =========================
# 1. Files and parameters
# =========================
npy_dir = Path(r"C:\Users\quzho2904\OneDrive - University of Bergen\GEOF337\report\npy")
fig_dir = Path(r"C:\Users\quzho2904\OneDrive - University of Bergen\GEOF337\report\figures")
fig_dir.mkdir(exist_ok=True)

file_list = [
    "HM2011622.npy",
    "GOS2012115.npy",
    "HM2013624.npy",
    "GOS2014117.npy",
    "HM2015620.npy",
    "HM2016619.npy",
    "GOS2017114.npy",
    "GOS2018111.npy",
    "GOS2019114.npy",
    "GOS2020112.npy",
    "DFN2021460.npy",
    "GOS2022112.npy",
    "GOS2023001013.npy",
    "GOS2024001014.npy",
    "GOS2025001014.npy",
    "HB2026009004.npy",
]

# Use only the specified station for each year
target_station_by_year = {
    2011: "992",
    2012: "444",
    2013: "1084",
    2014: "389",
    2015: "690",
    2016: "951",
    2017: "300",
    2018: "415",
    2019: "385",
    2020: "310",
    2021: "378",
    2022: "366",
    2023: "446",
    2024: "519",
    2025: "736",
    2026: "9",
}

# Depth range
zmin, zmax = 200.0, 400.0

# Output paths
out_csv = npy_dir / "selected_station_200_400m_timeseries_2011_2026.csv"
out_detail_csv = npy_dir / "selected_station_200_400m_detail_2011_2026.csv"

fig1_path = fig_dir / "selected_station_timeseries_O2_sigma0_200_400m_2011_2026.png"
fig2_path = fig_dir / "selected_station_timeseries_PT_S_200_400m_2011_2026.png"


# =========================
# 2. Utility functions
# =========================
def find_key(d, names):
    for n in names:
        if n in d:
            return n
    return None


def arr(x):
    return np.asarray(x, dtype=float)


def extract_year_from_filename(fname):
    m = re.search(r"(20\d{2})", fname)
    if m:
        return int(m.group(1))
    return np.nan


def get_lon_lat(st):
    lon_key = find_key(st, ["LON", "lon", "Longitude", "LONGITUDE"])
    lat_key = find_key(st, ["LAT", "lat", "Latitude", "LATITUDE"])
    if lon_key is None or lat_key is None:
        return np.nan, np.nan

    lon = arr(st[lon_key]).squeeze()
    lat = arr(st[lat_key]).squeeze()

    return float(lon), float(lat)


def get_depth(st):
    k = find_key(st, ["P", "Pressure", "pressure", "PRES", "pres"])
    if k is None:
        return None
    return arr(st[k])


def get_S_T(st):
    S_key = find_key(st, ["S", "SP", "salinity", "Salinity"])
    T_key = find_key(st, ["T", "TEMP", "temperature", "Temperature", "t"])
    if S_key is None or T_key is None:
        raise KeyError("Missing S or T")
    S = arr(st[S_key])
    T = arr(st[T_key])
    return S, T


def get_oxygen(st):
    k = find_key(st, ["raw_o", "OX", "oxygen", "OXYGEN", "DO"])
    if k is None:
        return None
    return arr(st[k])


def get_sigma0(st):
    k = find_key(st, ["SIGMA0", "sigma0", "Sigma0"])
    if k is None:
        return None
    return arr(st[k])


def fit_linear_stats(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    if np.sum(mask) < 2:
        return np.nan, np.nan, np.nan, np.nan

    res = linregress(x[mask], y[mask])

    slope = res.slope
    intercept = res.intercept
    r2 = res.rvalue ** 2
    pvalue = res.pvalue

    return slope, intercept, r2, pvalue


def find_station_key(ctd_dict, target_station):
    """
    Robustly find the target station ID in the dictionary.
    Compatible with keys stored as int, str, or strings with spaces.
    """
    target_station = str(target_station).strip()

    if target_station in ctd_dict:
        return target_station

    for k in ctd_dict.keys():
        if str(k).strip() == target_station:
            return k

    return None


def calculate_endpoint_rate(years, values, start_year, end_year):

    years = np.asarray(years, dtype=float)
    values = np.asarray(values, dtype=float)

    mask = (years >= start_year) & (years <= end_year) & np.isfinite(values)
    yrs_sel = years[mask]
    val_sel = values[mask]

    if len(yrs_sel) < 2:
        return np.nan, np.nan, np.nan, np.nan

    idx = np.argsort(yrs_sel)
    yrs_sel = yrs_sel[idx]
    val_sel = val_sel[idx]

    dyear = yrs_sel[-1] - yrs_sel[0]
    if dyear == 0:
        return np.nan, yrs_sel[0], yrs_sel[-1], np.nan

    rate = (val_sel[-1] - val_sel[0]) / dyear
    return rate, yrs_sel[0], yrs_sel[-1], (val_sel[-1] - val_sel[0])


# =========================
# 3. Single-station calculation
# =========================
def compute_station_means(st):
    lon, lat = get_lon_lat(st)

    z = get_depth(st)
    O_raw = get_oxygen(st)
    sigma0 = get_sigma0(st)

    if z is None or O_raw is None or sigma0 is None:
        return {
            "lon": lon,
            "lat": lat,
            "O2_200_400m_mLL": np.nan,
            "sigma0_200_400m": np.nan,
            "PT_200_400m_degC": np.nan,
            "S_200_400m": np.nan,
            "unit_flag": "missing_data",
            "status": "missing_data"
        }

    try:
        S, T = get_S_T(st)
    except Exception:
        return {
            "lon": lon,
            "lat": lat,
            "O2_200_400m_mLL": np.nan,
            "sigma0_200_400m": np.nan,
            "PT_200_400m_degC": np.nan,
            "S_200_400m": np.nan,
            "unit_flag": "missing_S_or_T",
            "status": "missing_S_or_T"
        }

    # Match array lengths
    n = min(len(z), len(O_raw), len(S), len(T), len(sigma0))

    z = z[:n]
    S = S[:n]
    T = T[:n]
    O_raw = O_raw[:n]
    sigma0 = sigma0[:n]

    valid_basic = (
        np.isfinite(z) &
        np.isfinite(S) &
        np.isfinite(T) &
        np.isfinite(O_raw) &
        np.isfinite(sigma0)
    )

    if np.sum(valid_basic) == 0:
        return {
            "lon": lon,
            "lat": lat,
            "O2_200_400m_mLL": np.nan,
            "sigma0_200_400m": np.nan,
            "PT_200_400m_degC": np.nan,
            "S_200_400m": np.nan,
            "unit_flag": "all_nan",
            "status": "all_nan"
        }

    z = z[valid_basic]
    S = S[valid_basic]
    T = T[valid_basic]
    O_raw = O_raw[valid_basic]
    sigma0 = sigma0[valid_basic]

    # ===== TEOS-10 =====
    try:
        SA = gsw.SA_from_SP(S, z, lon, lat)
        CT = gsw.CT_from_t(SA, T, z)
        rho = gsw.rho(SA, CT, z)   # kg/m3
        PT = gsw.pt0_from_t(SA, T, z)
    except Exception:
        return {
            "lon": lon,
            "lat": lat,
            "O2_200_400m_mLL": np.nan,
            "sigma0_200_400m": np.nan,
            "PT_200_400m_degC": np.nan,
            "S_200_400m": np.nan,
            "unit_flag": "gsw_failed",
            "status": "gsw_failed"
        }

    # ===== Determine oxygen unit =====
    O_mean_raw = np.nanmean(O_raw)

    if O_mean_raw < 20:
        O_mLL = O_raw
        unit_flag = "mL/L"
    else:
        rho_kgL = rho / 1000.0
        O_mLL = O_raw * rho_kgL / 44.66
        unit_flag = "umol/kg_to_mL/L"

    # ===== Mean over 200–400 m =====
    mask = (
        (z >= zmin) &
        (z <= zmax) &
        np.isfinite(O_mLL) &
        np.isfinite(sigma0) &
        np.isfinite(PT) &
        np.isfinite(S)
    )

    if np.sum(mask) == 0:
        return {
            "lon": lon,
            "lat": lat,
            "O2_200_400m_mLL": np.nan,
            "sigma0_200_400m": np.nan,
            "PT_200_400m_degC": np.nan,
            "S_200_400m": np.nan,
            "unit_flag": unit_flag,
            "status": "no_depth_data_200_400m"
        }

    return {
        "lon": lon,
        "lat": lat,
        "O2_200_400m_mLL": np.nanmean(O_mLL[mask]),
        "sigma0_200_400m": np.nanmean(sigma0[mask]),
        "PT_200_400m_degC": np.nanmean(PT[mask]),
        "S_200_400m": np.nanmean(S[mask]),
        "unit_flag": unit_flag,
        "status": "ok"
    }


# =========================
# 4. Main loop: only extract the specified station each year
# =========================
summary = []
detail = []

for fname in file_list:
    fpath = npy_dir / fname
    year = extract_year_from_filename(fname)
    target_station = target_station_by_year.get(year, None)

    if not fpath.exists():
        print(f"\nMissing file: {fname}")
        summary.append({
            "file": fname,
            "year": year,
            "target_station": target_station,
            "station_found": False,
            "lon": np.nan,
            "lat": np.nan,
            "O2_200_400m_mean_mLL": np.nan,
            "sigma0_200_400m_mean": np.nan,
            "PT_200_400m_mean_degC": np.nan,
            "S_200_400m_mean": np.nan,
            "unit_flag": "file_missing",
            "status": "file_missing"
        })
        continue

    if target_station is None:
        print(f"\n{fname}: No target station set for this year")
        summary.append({
            "file": fname,
            "year": year,
            "target_station": np.nan,
            "station_found": False,
            "lon": np.nan,
            "lat": np.nan,
            "O2_200_400m_mean_mLL": np.nan,
            "sigma0_200_400m_mean": np.nan,
            "PT_200_400m_mean_degC": np.nan,
            "S_200_400m_mean": np.nan,
            "unit_flag": "no_target_station_for_year",
            "status": "no_target_station_for_year"
        })
        continue

    ctd = np.load(fpath, allow_pickle=True).item()
    station_key = find_station_key(ctd, target_station)

    if station_key is None:
        print(f"\n{fname}: Target station {target_station} not found")
        detail.append({
            "file": fname,
            "year": year,
            "target_station": target_station,
            "station": np.nan,
            "station_found": False,
            "lon": np.nan,
            "lat": np.nan,
            "O2_200_400m_mLL": np.nan,
            "sigma0_200_400m": np.nan,
            "PT_200_400m_degC": np.nan,
            "S_200_400m": np.nan,
            "unit_flag": "station_not_found",
            "status": "station_not_found"
        })

        summary.append({
            "file": fname,
            "year": year,
            "target_station": target_station,
            "station_found": False,
            "lon": np.nan,
            "lat": np.nan,
            "O2_200_400m_mean_mLL": np.nan,
            "sigma0_200_400m_mean": np.nan,
            "PT_200_400m_mean_degC": np.nan,
            "S_200_400m_mean": np.nan,
            "unit_flag": "station_not_found",
            "status": "station_not_found"
        })
        continue

    st = ctd[station_key]
    result = compute_station_means(st)

    detail.append({
        "file": fname,
        "year": year,
        "target_station": target_station,
        "station": str(station_key),
        "station_found": True,
        "lon": result["lon"],
        "lat": result["lat"],
        "O2_200_400m_mLL": result["O2_200_400m_mLL"],
        "sigma0_200_400m": result["sigma0_200_400m"],
        "PT_200_400m_degC": result["PT_200_400m_degC"],
        "S_200_400m": result["S_200_400m"],
        "unit_flag": result["unit_flag"],
        "status": result["status"]
    })

    summary.append({
        "file": fname,
        "year": year,
        "target_station": target_station,
        "station_found": True,
        "lon": result["lon"],
        "lat": result["lat"],
        "O2_200_400m_mean_mLL": result["O2_200_400m_mLL"],
        "sigma0_200_400m_mean": result["sigma0_200_400m"],
        "PT_200_400m_mean_degC": result["PT_200_400m_degC"],
        "S_200_400m_mean": result["S_200_400m"],
        "unit_flag": result["unit_flag"],
        "status": result["status"]
    })


# =========================
# 5. Save tables
# =========================
df_summary = pd.DataFrame(summary)
df_detail = pd.DataFrame(detail)

df_summary = df_summary.sort_values("year").reset_index(drop=True)
df_detail = df_detail.sort_values(["year"]).reset_index(drop=True)

df_summary.to_csv(out_csv, index=False, encoding="utf-8-sig")
df_detail.to_csv(out_detail_csv, index=False, encoding="utf-8-sig")

print("\nSaved successfully:")
print(out_csv)
print(out_detail_csv)


# =========================
# 6. Trend fitting
# =========================
years = df_summary["year"].values.astype(float)

O2 = df_summary["O2_200_400m_mean_mLL"].values.astype(float)
sigma0 = df_summary["sigma0_200_400m_mean"].values.astype(float)
PT = df_summary["PT_200_400m_mean_degC"].values.astype(float)
S = df_summary["S_200_400m_mean"].values.astype(float)

# Figure 1: sigma0 trend for 2011–2026
mask_fit_sigma = (years >= 2011) & (years <= 2026) & np.isfinite(sigma0)
sigma_slope, sigma_intercept, sigma_r2, sigma_p = fit_linear_stats(
    years[mask_fit_sigma],
    sigma0[mask_fit_sigma]
)

# Figure 2: PT and S trends for 2011–2026
mask_fit_PT = (years >= 2011) & (years <= 2026) & np.isfinite(PT)
PT_slope, PT_intercept, PT_r2, PT_p = fit_linear_stats(
    years[mask_fit_PT],
    PT[mask_fit_PT]
)

mask_fit_S = (years >= 2011) & (years <= 2026) & np.isfinite(S)
S_slope, S_intercept, S_r2, S_p = fit_linear_stats(
    years[mask_fit_S],
    S[mask_fit_S]
)

print("\nRegression results for 2011–2026:")
print(f"sigma0 slope = {sigma_slope:.6f} /yr, R2 = {sigma_r2:.4f}, p = {sigma_p:.4f}")
print(f"PT slope     = {PT_slope:.6f} degC/yr, R2 = {PT_r2:.4f}, p = {PT_p:.4f}")
print(f"S slope      = {S_slope:.6f} /yr, R2 = {S_r2:.4f}, p = {S_p:.4f}")


# =========================
# 7. Direct calculation of annual change
#    Periods: 2013–2017, 2021–2024
# =========================
periods = [
    (2013, 2017),
    (2021, 2024),
]

# Save change rates for the two periods
O2_rate_list = []
sigma_rate_list = []

print("\n==============================")
print("Direct annual change (endpoint method)")
print("==============================")

for start_year, end_year in periods:
    dO2_dt, y1_o2, y2_o2, dO2 = calculate_endpoint_rate(years, O2, start_year, end_year)
    dsigma_dt, y1_sg, y2_sg, dsigma = calculate_endpoint_rate(years, sigma0, start_year, end_year)

    print(f"\nPeriod: {start_year}–{end_year}")

    if np.isfinite(dO2_dt):
        print(f"Change in oxygen (mL O2 L^-1 yr^-1): {dO2_dt:.6f}")
        print(f"Total oxygen change over period:      {dO2:.6f}")
        O2_rate_list.append(dO2_dt)
    else:
        print("Change in oxygen: insufficient data")

    if np.isfinite(dsigma_dt):
        print(f"Annual change in sigma0:              {dsigma_dt:.6f}")
        print(f"Total sigma0 change over period:      {dsigma:.6f}")
        sigma_rate_list.append(dsigma_dt)
    else:
        print("Annual change in sigma0: insufficient data")

# ===== Mean change rate across the two periods =====
if len(O2_rate_list) > 0:
    mean_dO2_dt = np.nanmean(O2_rate_list)
    mean_abs_dO2_dt = np.nanmean(np.abs(O2_rate_list))

    print("\n------------------------------")
    print("Mean oxygen annual change from the two periods:")
    print(f"Signed mean = {mean_dO2_dt:.6f} mL O2 L^-1 yr^-1")
    print(f"Absolute mean = {mean_abs_dO2_dt:.6f} mL O2 L^-1 yr^-1")
else:
    mean_dO2_dt = np.nan
    mean_abs_dO2_dt = np.nan
    print("\nMean oxygen annual change: insufficient data")

if len(sigma_rate_list) > 0:
    mean_dsigma_dt = np.nanmean(sigma_rate_list)
    mean_abs_dsigma_dt = np.nanmean(np.abs(sigma_rate_list))

    print("\nMean sigma0 annual change from the two periods:")
    print(f"Signed mean = {mean_dsigma_dt:.6f} yr^-1")
    print(f"Absolute mean = {mean_abs_dsigma_dt:.6f} yr^-1")
else:
    mean_dsigma_dt = np.nan
    mean_abs_dsigma_dt = np.nan
    print("\nMean sigma0 annual change: insufficient data")

print("==============================")

# =========================
# 8. Plot 1: O2 + sigma0
# =========================
fig, ax1 = plt.subplots(figsize=(9, 6), dpi=200)

# O2 time series
mask1 = np.isfinite(years) & np.isfinite(O2)
ax1.plot(
    years[mask1], O2[mask1],
    marker="o",
    linestyle="-",
    linewidth=1.8,
    color="tab:blue",
    label="O2"
)

ax1.set_xlabel("Year")
ax1.set_ylabel("O2 (mL/L)", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

# sigma0
ax2 = ax1.twinx()

mask2 = np.isfinite(years) & np.isfinite(sigma0)
ax2.plot(
    years[mask2], sigma0[mask2],
    marker="o",
    linestyle="-",
    linewidth=1.8,
    color="tab:red",
    label="Sigma0"
)

# sigma0 trend
if np.sum(mask_fit_sigma) >= 2:
    xfit_sigma = years[mask_fit_sigma]
    sort_idx = np.argsort(xfit_sigma)
    xfit_sigma = xfit_sigma[sort_idx]
    yfit_sigma = sigma_slope * xfit_sigma + sigma_intercept

    ax2.plot(
        xfit_sigma,
        yfit_sigma,
        linestyle=":",
        linewidth=1.8,
        color="tab:red",
        label=f"Sigma0 trend (2011–2026): slope={sigma_slope:.4f}, R²={sigma_r2:.3f}, p={sigma_p:.4f}"
    )

ax2.set_ylabel("Sigma0", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="best",
    fontsize=9
)

plt.title("Masfjorden selected-station O2 and Sigma0 (200–400 m)")
plt.tight_layout()
plt.savefig(fig1_path, dpi=300, bbox_inches="tight")
plt.close()


# =========================
# 9. Plot 2: PT + S
#    Both with 2011–2026 trends
# =========================
fig, ax1 = plt.subplots(figsize=(9, 6), dpi=200)

# PT
mask3 = np.isfinite(years) & np.isfinite(PT)
ax1.plot(
    years[mask3], PT[mask3],
    marker="o",
    linestyle="-",
    linewidth=1.8,
    color="tab:green",
    label="Potential Temperature"
)

# PT trend
if np.sum(mask_fit_PT) >= 2:
    xfit_PT = years[mask_fit_PT]
    sort_idx = np.argsort(xfit_PT)
    xfit_PT = xfit_PT[sort_idx]
    yfit_PT = PT_slope * xfit_PT + PT_intercept

    ax1.plot(
        xfit_PT,
        yfit_PT,
        linestyle="--",
        linewidth=1.5,
        color="tab:green",
        label=f"PT trend (2011–2026): slope={PT_slope:.4f}, R²={PT_r2:.3f}, p={PT_p:.4f}"
    )

ax1.set_xlabel("Year")
ax1.set_ylabel("Potential Temperature (°C)", color="tab:green")
ax1.tick_params(axis="y", labelcolor="tab:green")

# S
ax2 = ax1.twinx()

mask4 = np.isfinite(years) & np.isfinite(S)
ax2.plot(
    years[mask4], S[mask4],
    marker="o",
    linestyle="-",
    linewidth=1.8,
    color="tab:purple",
    label="Salinity"
)

# S trend
if np.sum(mask_fit_S) >= 2:
    xfit_S = years[mask_fit_S]
    sort_idx = np.argsort(xfit_S)
    xfit_S = xfit_S[sort_idx]
    yfit_S = S_slope * xfit_S + S_intercept

    ax2.plot(
        xfit_S,
        yfit_S,
        linestyle=":",
        linewidth=1.5,
        color="tab:purple",
        label=f"S trend (2011–2026): slope={S_slope:.4f}, R²={S_r2:.3f}, p={S_p:.4f}"
    )

ax2.set_ylabel("Salinity", color="tab:purple")
ax2.tick_params(axis="y", labelcolor="tab:purple")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="best",
    fontsize=9
)

plt.title("Masfjorden selected-station Potential Temperature and Salinity (200–400 m)")
plt.tight_layout()
plt.savefig(fig2_path, dpi=300, bbox_inches="tight")
plt.close()

print("\nFigures saved:")
print(fig1_path)
print(fig2_path)

# =========================
# 10. Calculate annual Vn
# =========================

Os = 5.4
delta_t = 1.0

O2_series = df_summary["O2_200_400m_mean_mLL"].values.astype(float)
years = df_summary["year"].values.astype(float)

Vn_list = [np.nan]  # no previous year for the first year

for i in range(1, len(O2_series)):

    OB_n = O2_series[i]
    OB_prev = O2_series[i-1]

    if not (np.isfinite(OB_n) and np.isfinite(OB_prev)):
        Vn_list.append(np.nan)
        continue

    numerator = OB_n - OB_prev + mean_abs_dO2_dt * delta_t
    denominator = (Os - OB_prev) * delta_t

    if denominator == 0:
        Vn = np.nan
    else:
        Vn = numerator / denominator

    Vn_list.append(Vn)


df_summary["Vn"] = Vn_list


print("\nVn calculation complete")
print("b used =", mean_abs_dO2_dt)
print("Os =", Os)

print("\nVn preview:")
print(df_summary[["year", "O2_200_400m_mean_mLL", "Vn"]])

# =========================
# Save Vn time series
# =========================

out_vn_csv = npy_dir / "Vn_timeseries_2011_2026.csv"

df_vn = df_summary[
    ["year", "O2_200_400m_mean_mLL", "Vn"]
].copy()

df_vn.columns = [
    "year",
    "O2_200_400m_mLL",
    "Vn"
]

df_vn.to_csv(
    out_vn_csv,
    index=False,
    encoding="utf-8-sig"
)

print("\nVn saved:")
print(out_vn_csv)


Saved successfully:
C:\Users\quzho2904\OneDrive - University of Bergen\GEOF337\report\npy\selected_station_200_400m_timeseries_2011_2026.csv
C:\Users\quzho2904\OneDrive - University of Bergen\GEOF337\report\npy\selected_station_200_400m_detail_2011_2026.csv

Regression results for 2011–2026:
sigma0 slope = -0.008680 /yr, R2 = 0.7773, p = 0.0000
PT slope     = 0.011593 degC/yr, R2 = 0.5025, p = 0.0021
S slope      = -0.008807 /yr, R2 = 0.6090, p = 0.0004

Direct annual change (endpoint method)

Period: 2013–2017
Change in oxygen (mL O2 L^-1 yr^-1): -0.434415
Total oxygen change over period:      -1.737661
Annual change in sigma0:              -0.014949
Total sigma0 change over period:      -0.059796

Period: 2021–2024
Change in oxygen (mL O2 L^-1 yr^-1): -0.270549
Total oxygen change over period:      -0.811648
Annual change in sigma0:              -0.017994
Total sigma0 change over period:      -0.053981

------------------------------
Mean oxygen annual change from the two periods:
S